# Thực nghiệm Flowers Recognition: RepLKNet (2022) và VGG-16

Đây là thực nghiệm transfer learning trên cùng một dataset, cùng split và cùng protocol. VGG-16 đại diện cho CNN thuần truyền thống; RepLKNet-31B đại diện cho large-kernel CNN. Kết quả test được giữ độc lập với validation và lưu thành artifacts trong `results/flowers/`.

## Cấu hình thực nghiệm

Run chính thức dùng ba seed để báo cáo mean ± standard deviation. Với RTX 3050 6 GB, batch size 8 được giữ cố định để cả hai model chạy được trên cùng phần cứng.

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'flower_experiment.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from flower_experiment import run_experiment

DATA_DIR = PROJECT_ROOT / 'data' / 'flowers'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'flowers'
CHECKPOINT = PROJECT_ROOT / 'weights' / 'RepLKNet-31B_ImageNet-1K_224.pth'
SEEDS = [42, 123, 2024]
EPOCHS = 10
BATCH_SIZE = 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('CUDA không khả dụng. Hãy chọn kernel Python (RepLKNet Demo) và kiểm tra lại PyTorch CUDA.')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## Chạy thực nghiệm

Các model đều khởi tạo từ ImageNet pretrained backbone, thay classifier cuối bằng 5 lớp hoa. RepLKNet chỉ được structural re-parameterization sau khi chọn best checkpoint trên validation.

In [ ]:
summary_rows = run_experiment(
    data_dir=DATA_DIR,
    project_root=PROJECT_ROOT,
    output_dir=OUTPUT_DIR,
    replk_checkpoint=CHECKPOINT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seeds=SEEDS,
    num_workers=0,
    device_name=DEVICE,
    deterministic=True,
)

## Tổng hợp metrics

Bảng dưới đây lấy trực tiếp từ test split đã khóa trong `split_manifest.json`; không lấy số liệu từ training hoặc validation để thay thế.

In [ ]:
import json
from IPython.display import HTML, display

aggregate = json.loads((OUTPUT_DIR / 'aggregate_metrics.json').read_text(encoding='utf-8'))
rows = []
for model, values in aggregate.items():
    rows.append({
        'model': model,
        'accuracy': values['test_accuracy'],
        'macro_f1': values['test_macro_f1'],
        'balanced_accuracy': values['test_balanced_accuracy'],
        'latency_ms': values['latency_mean_ms'],
    })
headers = ['Model', 'Accuracy', 'Macro-F1', 'Balanced accuracy', 'Latency (ms)']
html = '<table><thead><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr></thead><tbody>'
for row in rows:
    html += '<tr>' + ''.join([
        f"<td>{row['model']}</td>",
        f"<td>{row['accuracy']['mean']:.4f} ± {row['accuracy']['std']:.4f}</td>",
        f"<td>{row['macro_f1']['mean']:.4f} ± {row['macro_f1']['std']:.4f}</td>",
        f"<td>{row['balanced_accuracy']['mean']:.4f} ± {row['balanced_accuracy']['std']:.4f}</td>",
        f"<td>{row['latency_ms']['mean']:.3f}</td>",
    ]) + '</tr>'
html += '</tbody></table>'
display(HTML(html))
print((OUTPUT_DIR / 'conclusion.md').read_text(encoding='utf-8'))